<a href="https://colab.research.google.com/github/mofermino/condensed-subject-matters/blob/main/fit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: File Upload
from google.colab import files

print("Please upload your data file (.txt, .csv, etc.)...")
uploaded = files.upload()

# Get the name of the uploaded file
if uploaded:
    file_name = next(iter(uploaded))
    print(f"\nSuccessfully uploaded '{file_name}'!")
else:
    print("\nNo file uploaded.")

Please upload your data file (.txt, .csv, etc.)...


Saving D1_Rxy_H_1uA_Set_Temp_2.txt to D1_Rxy_H_1uA_Set_Temp_2.txt

Successfully uploaded 'D1_Rxy_H_1uA_Set_Temp_2.txt'!


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Hall bar data Bézier fitting

Features
--------
1) Loads a table containing (at least) the real/imag components from a lock-in amplifier,
   the AC current amplitude and frequency, and a sweep variable (magnetic field or temperature).
2) Computes V magnitude/phase and R = V/I channels.
3) Supports filtering by measurement type: longitudinal / transversal (transverse).
4) Fits data with:
   a) Functional Bézier via Bernstein basis (best when y is a single-valued function of x).
   b) Piecewise cubic, parametric Bézier spline (best for hysteresis loops or multi-valued y(x)).
5) Saves plots and (optionally) fitted curve samples to CSV.

Usage examples
--------------
# Functional Bézier fit (degree 7) to R_real vs magnetic field, longitudinal only:
python hall_bezier_fit.py --file data.csv --measurement longitudinal \
    --y R_real --method bernstein --degree 7 --plot fit.png --export fit.csv

# Parametric cubic Bézier spline for a hysteresis loop (R_mag vs field):
python hall_bezier_fit.py --file data.csv --y R_mag --method parametric \
    --segment-len 60 --points-per-segment 80 --plot loop_fit.png

Notes
-----
- Column auto-detection tries common names. Use --xcol/--real/--imag/--icurr/--freq/--meas to override.
- Measurement labels accepted: longitudinal|transversal|transverse|Rxx|Rxy (case-insensitive).
- If your dataset uses temperature instead of field as the sweep variable, set --xcol to that column.
"""



from __future__ import annotations
import argparse
from dataclasses import dataclass
from math import comb
from typing import Dict, List, Optional, Tuple

from __future__ import annotations
from dataclasses import dataclass
from math import comb
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -----------------------------
# Utilities: column detection
# -----------------------------
def _first_present(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]
    return None

@dataclass
class ColumnMap:
    x: str
    v_real: Optional[str]
    v_imag: Optional[str]
    i_ac: Optional[str]
    freq: Optional[str]
    meas: Optional[str]  # measurement label column (longitudinal/transversal/transverse/Rxx/Rxy)


def detect_columns(df: pd.DataFrame) -> ColumnMap:
    # X axis: magnetic field or temperature (cooling)
    x_candidates = [
        "H", "Field", "B", "mu0H", "magnetic_field", "field_T", "H_T",
        "Temperature", "Temp", "T", "K", "temperature_K", "applied_cooling"
    ]
    x = _first_present(df, x_candidates)
    if x is None:
        # Fallback: try the first numeric column
        numeric_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
        if not numeric_cols:
            raise ValueError("Could not infer sweep column (field/temperature). Please specify --xcol.")
        x = numeric_cols[0]

    # Lock-in channels (real/imag)
    v_real = _first_present(df, ["V_real", "Vx", "in_phase", "X", "V_real_V", "V1X", "Re"])
    v_imag = _first_present(df, ["V_imag", "Vy", "quadrature", "Y", "V_imag_V", "V1Y", "Im"])

    # Current and frequency
    i_ac = _first_present(df, ["I", "I_ac", "Iac", "I_rms", "current", "current_A", "AC_current"])
    freq = _first_present(df, ["f", "freq", "frequency", "Hz"])

    # Measurement label (orientation/channel)
    meas = _first_present(df, ["measurement", "type", "orientation", "channel", "meas", "config"])

    return ColumnMap(x=x, v_real=v_real, v_imag=v_imag, i_ac=i_ac, freq=freq, meas=meas)


# -----------------------------
# Compute derived channels
# -----------------------------

def add_derived_channels(df: pd.DataFrame, cm: ColumnMap) -> pd.DataFrame:
    out = df.copy()

    # Voltage magnitude & phase (if both real/imag exist)
    if cm.v_real is not None and cm.v_imag is not None:
        out["V_real"] = pd.to_numeric(out[cm.v_real], errors="coerce")
        out["V_imag"] = pd.to_numeric(out[cm.v_imag], errors="coerce")
        out["V_mag"] = np.sqrt(out["V_real"]**2 + out["V_imag"]**2)
        out["V_phase_rad"] = np.arctan2(out["V_imag"], out["V_real"])
    elif cm.v_real is not None:
        out["V_real"] = pd.to_numeric(out[cm.v_real], errors="coerce")
        out["V_imag"] = np.nan
        out["V_mag"] = out["V_real"].abs()
        out["V_phase_rad"] = np.nan
    else:
        raise ValueError("Need at least a real voltage column. Use --real to specify it.")

    # Current (may be a scalar column or constant missing -> assume 1 A if unknown)
    if cm.i_ac is not None:
        out["I_ac"] = pd.to_numeric(out[cm.i_ac], errors="coerce")
    else:
        out["I_ac"] = 1.0  # Fallback (so R == V if current unknown)

    # Resistances
    out["R_real"] = out["V_real"] / out["I_ac"]
    out["R_imag"] = out["V_imag"] / out["I_ac"]
    out["R_mag"] = out["V_mag"] / out["I_ac"]

    # Normalize measurement labels if present
    if cm.meas is not None:
        m = out[cm.meas].astype(str).str.lower()
        # map common aliases
        m = m.replace({
            "rxx": "longitudinal",
            "rxy": "transversal",
            "transverse": "transversal",
        })
        out["_measurement_norm"] = m
    else:
        out["_measurement_norm"] = "unknown"

    # Ensure x is numeric
    out["_x"] = pd.to_numeric(out[cm.x], errors="coerce")

    # Drop rows with missing x or y later; for now just return
    return out


# -----------------------------
# Bézier via Bernstein (functional y(x))
# -----------------------------

def bernstein_matrix(n: int, t: np.ndarray) -> np.ndarray:
    """Return the (N, n+1) Bernstein basis matrix for degree n at samples t in [0,1]."""
    B = np.empty((t.size, n + 1), dtype=float)
    # Vectorized computation for stability
    # B_k(t) = C(n,k) t^k (1-t)^(n-k)
    one_minus_t = 1.0 - t
    # Precompute powers
    t_pows = np.vstack([t**k for k in range(n + 1)]).T  # (N, n+1)
    omt_pows = np.vstack([one_minus_t**(n - k) for k in range(n + 1)]).T
    for k in range(n + 1):
        B[:, k] = comb(n, k) * t_pows[:, k] * omt_pows[:, k]
    return B


def fit_bernstein(x: np.ndarray, y: np.ndarray, degree: int = 5, l2: float = 0.0
                  ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Fit y(x) with an n-th degree Bernstein polynomial (Bézier) on normalized x in [0,1].

    Returns
    -------
    t_eval : (M,) array of evaluation points in [0,1]
    y_fit  : (M,) fitted values
    ctrl_y : (degree+1,) control "y" coordinates (control x are fixed at k/n)
    """
    # Sort by x to make a proper function y(x)
    idx = np.argsort(x)
    xx = np.asarray(x, float)[idx]
    yy = np.asarray(y, float)[idx]

    xmin, xmax = float(np.nanmin(xx)), float(np.nanmax(xx))
    if xmax <= xmin:
        raise ValueError("x has zero span; cannot normalize.")
    t = (xx - xmin) / (xmax - xmin)

    mask = np.isfinite(t) & np.isfinite(yy)
    t = t[mask]
    yy = yy[mask]

    B = bernstein_matrix(degree, t)
    if l2 > 0:
        # Ridge regularized normal equations
        BTB = B.T @ B
        BTy = B.T @ yy
        ctrl_y = np.linalg.solve(BTB + l2 * np.eye(degree + 1), BTy)
    else:
        ctrl_y, *_ = np.linalg.lstsq(B, yy, rcond=None)

    # Evaluate on a fine grid
    t_eval = np.linspace(0, 1, 1000)
    B_eval = bernstein_matrix(degree, t_eval)
    y_fit = B_eval @ ctrl_y
    return t_eval * (xmax - xmin) + xmin, y_fit, ctrl_y


# -----------------------------
# Piecewise cubic parametric Bézier (good for loops)
# -----------------------------

def chord_length_parameterize(points: np.ndarray) -> np.ndarray:
    """Return t in [0,1] by cumulative chord length through 2D points."""
    diffs = np.diff(points, axis=0)
    seg = np.sqrt((diffs**2).sum(axis=1))
    s = np.concatenate([[0.0], np.cumsum(seg)])
    if s[-1] == 0:
        return np.linspace(0, 1, len(points))
    return s / s[-1]


def fit_cubic_bezier_segment(points: np.ndarray) -> np.ndarray:
    """
    Fit a single cubic Bézier segment to a set of 2D points by least squares.

    Returns 4x2 array of control points [P0, P1, P2, P3].
    """
    n = points.shape[0]
    if n < 4:
        raise ValueError("Need at least 4 points to fit a cubic Bézier segment.")

    P0 = points[0]
    P3 = points[-1]
    t = chord_length_parameterize(points)

    # Bernstein basis for cubic
    B0 = (1 - t)**3
    B1 = 3 * t * (1 - t)**2
    B2 = 3 * t**2 * (1 - t)
    B3 = t**3

    # Build least-squares system for P1 and P2
    M = np.column_stack([B1, B2])  # (n,2)
    rhs = points - (B0[:, None] * P0 + B3[:, None] * P3)  # (n,2)

    # Solve separately for x and y
    # Guard against singular matrices using lstsq
    P12x, *_ = np.linalg.lstsq(M, rhs[:, 0], rcond=None)
    P12y, *_ = np.linalg.lstsq(M, rhs[:, 1], rcond=None)
    P1 = np.array([P12x[0], P12y[0]])
    P2 = np.array([P12x[1], P12y[1]])

    CP = np.vstack([P0, P1, P2, P3])
    return CP


def eval_cubic_bezier(CP: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Evaluate cubic Bézier with 4x2 control points CP at parameter t in [0,1]."""
    B0 = (1 - t)**3
    B1 = 3 * t * (1 - t)**2
    B2 = 3 * t**2 * (1 - t)
    B3 = t**3
    return (B0[:, None] * CP[0] +
            B1[:, None] * CP[1] +
            B2[:, None] * CP[2] +
            B3[:, None] * CP[3])


def fit_piecewise_bezier(x: np.ndarray,
                         y: np.ndarray,
                         segment_len: int = 50,
                         points_per_segment: int = 80) -> Tuple[np.ndarray, np.ndarray, List[np.ndarray]]:
    """
    Fit a piecewise cubic parametric Bézier spline to (x, y) data.

    Parameters
    ----------
    segment_len : number of raw points per segment (>= 4)
    points_per_segment : number of evaluated points per segment

    Returns
    -------
    x_fit, y_fit : concatenated evaluated spline
    segments     : list of 4x2 control point arrays for each segment
    """
    if segment_len < 4:
        raise ValueError("segment_len must be at least 4.")

    # Maintain the acquisition order (important for loops)
    xx = np.asarray(x, float)
    yy = np.asarray(y, float)
    mask = np.isfinite(xx) & np.isfinite(yy)
    xx, yy = xx[mask], yy[mask]

    pts = np.column_stack([xx, yy])
    N = pts.shape[0]
    segments: List[np.ndarray] = []
    xs, ys = [], []

    # Overlap neighbor segments by 1 point to enforce continuity at joints
    i = 0
    while i < N - 1:
        j = min(i + segment_len, N)
        seg_pts = pts[i:j]
        if seg_pts.shape[0] < 4:
            # For the tail, append last point and break
            if xs and ys:
                xs.append([pts[-1, 0]])
                ys.append([pts[-1, 1]])
            break
        CP = fit_cubic_bezier_segment(seg_pts)
        segments.append(CP)
        t_eval = np.linspace(0, 1, points_per_segment)
        eval_pts = eval_cubic_bezier(CP, t_eval)
        # Avoid duplicate at the boundary: skip the first point for all but the first segment
        if i > 0:
            eval_pts = eval_pts[1:]
        xs.append(eval_pts[:, 0])
        ys.append(eval_pts[:, 1])
        # Advance with an overlap of 1 raw point
        i = j - 1

    x_fit = np.concatenate(xs) if xs else np.array([])
    y_fit = np.concatenate(ys) if ys else np.array([])
    return x_fit, y_fit, segments


# -----------------------------
# Plotting
# -----------------------------

def make_plot(x: np.ndarray,
              y: np.ndarray,
              x_fit: np.ndarray,
              y_fit: np.ndarray,
              title: str,
              xlabel: str,
              ylabel: str,
              save_path: Optional[str] = None,
              show: bool = False) -> None:
    plt.figure(figsize=(7.5, 5.0))
    plt.scatter(x, y, s=12, alpha=0.6, label="Data")
    plt.plot(x_fit, y_fit, linewidth=2.0, label="Bézier fit")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200)
    if show:
        plt.show()
    plt.close()


# -----------------------------
# Main driver
# -----------------------------

def main():
    ap = argparse.ArgumentParser(description="Bézier curve fitting for Hall bar measurements.")
    ap.add_argument("--file", required=True, help="Input data file (CSV/TSV/XLSX).")
    ap.add_argument("--xcol", default=None, help="Name of sweep column (field/temperature).")
    ap.add_argument("--real", default=None, help="Name of real voltage column (lock-in X).")
    ap.add_argument("--imag", default=None, help="Name of imaginary voltage column (lock-in Y).")
    ap.add_argument("--icurr", default=None, help="Name of AC current column (A).")
    ap.add_argument("--freq", default=None, help="Name of frequency column (Hz).")
    ap.add_argument("--meas", default=None, help="Name of measurement label column (e.g., Rxx/Rxy).")
    ap.add_argument("--measurement", default="all",
                    choices=["all", "longitudinal", "transversal"],
                    help="Filter data by measurement type (case-insensitive).")
    ap.add_argument("--y", default="R_real",
                    choices=["V_real", "V_imag", "V_mag", "R_real", "R_imag", "R_mag"],
                    help="Which column to fit.")
    ap.add_argument("--method", default="bernstein", choices=["bernstein", "parametric"],
                    help="Bézier fitting method.")
    ap.add_argument("--degree", type=int, default=5, help="Degree for Bernstein (functional) fit.")
    ap.add_argument("--lambda", dest="ridge", type=float, default=0.0,
                    help="L2 regularization (ridge) for Bernstein fit.")
    ap.add_argument("--segment-len", type=int, default=50, help="Raw points per segment (parametric).")
    ap.add_argument("--points-per-segment", type=int, default=80,
                    help="Evaluated points per segment (parametric).")
    ap.add_argument("--plot", default=None, help="Path to save PNG plot.")
    ap.add_argument("--export", default=None, help="Path to save fitted curve CSV.")
    ap.add_argument("--show", action="store_true", help="Show plot interactively.")
    args = ap.parse_args()

    # Load
    ext = args.file.lower().split(".")[-1]
    if ext in ("xls", "xlsx"):
        df = pd.read_excel(args.file)
    else:
        # Let pandas infer the delimiter; works for CSV/TSV
        df = pd.read_csv(args.file)

    # Detect columns
    cm = detect_columns(df)
    # Override if provided
    if args.xcol: cm.x = args.xcol
    if args.real: cm.v_real = args.real
    if args.imag: cm.v_imag = args.imag
    if args.icurr: cm.i_ac = args.icurr
    if args.freq: cm.freq = args.freq
    if args.meas: cm.meas = args.meas

    # Derived channels
    dfp = add_derived_channels(df, cm)

    # Filter by measurement type if requested
    if args.measurement != "all":
        sel = dfp["_measurement_norm"] == args.measurement
        if not np.any(sel):
            raise ValueError(f"No rows matching measurement '{args.measurement}'. "
                             f"Available (normalized) labels: "
                             f"{sorted(dfp['_measurement_norm'].unique().tolist())}")
        dfp = dfp.loc[sel].copy()

    # Pull x and y
    if args.y not in dfp.columns:
        raise ValueError(f"Requested y '{args.y}' not found after preprocessing.")
    if "_x" not in dfp.columns:
        raise ValueError("Internal x column missing after preprocessing.")

    x = dfp["_x"].to_numpy()
    y = dfp[args.y].to_numpy()

    # Do the fit
    title = f"{args.method.capitalize()} Bézier fit: {args.y} vs {cm.x}"
    xlabel = cm.x
    ylabel = args.y

    if args.method == "bernstein":
        x_fit, y_fit, ctrl_y = fit_bernstein(x, y, degree=args.degree, l2=args.ridge)
        # Optional: print control points (x positions are k/n scaled to x-range)
        x_min, x_max = np.nanmin(x), np.nanmax(x)
        ctrl_x = np.linspace(x_min, x_max, args.degree + 1)
        print("# Bernstein control points (x, y):")
        for xi, yi in zip(ctrl_x, ctrl_y):
            print(f"{xi:.6g}, {yi:.6g}")

    else:  # parametric
        x_fit, y_fit, segments = fit_piecewise_bezier(
            x, y, segment_len=args.segment_len, points_per_segment=args.points_per_segment
        )
        print(f"# Fitted {len(segments)} cubic Bézier segment(s).")
        # Optionally, print end points of segments
        for si, CP in enumerate(segments, 1):
            P0, P1, P2, P3 = CP
            print(f"  Segment {si}: P0={P0}, P1={P1}, P2={P2}, P3={P3}")

    # Plot
    make_plot(x, y, x_fit, y_fit, title=title, xlabel=xlabel, ylabel=ylabel,
              save_path=args.plot, show=args.show)

    # Export fitted curve
    if args.export:
        out_df = pd.DataFrame({cm.x: x_fit, f"{args.y}_fit": y_fit})
        out_df.to_csv(args.export, index=False)
        print(f"Saved fitted curve to: {args.export}")
    if args.plot:
        print(f"Saved plot to: {args.plot}")


if __name__ == "__main__":
    main()


usage: colab_kernel_launcher.py [-h] --file FILE [--xcol XCOL] [--real REAL]
                                [--imag IMAG] [--icurr ICURR] [--freq FREQ]
                                [--meas MEAS]
                                [--measurement {all,longitudinal,transversal}]
                                [--y {V_real,V_imag,V_mag,R_real,R_imag,R_mag}]
                                [--method {bernstein,parametric}]
                                [--degree DEGREE] [--lambda RIDGE]
                                [--segment-len SEGMENT_LEN]
                                [--points-per-segment POINTS_PER_SEGMENT]
                                [--plot PLOT] [--export EXPORT] [--show]
colab_kernel_launcher.py: error: the following arguments are required: --file


SystemExit: 2

In [ ]:
# Cell 1: File Upload
from google.colab import files

print("Please upload your data file (.txt, .csv, etc.)...")
uploaded = files.upload()

# Get the name of the uploaded file
if uploaded:
    file_name = next(iter(uploaded))
    print(f"\nSuccessfully uploaded '{file_name}'!")
else:
    print("\nNo file uploaded.")

Please upload your data file (.txt, .csv, etc.)...


Saving D1_Rxx_H_1uA_Set_Temp_22.txt to D1_Rxx_H_1uA_Set_Temp_22.txt

Successfully uploaded 'D1_Rxx_H_1uA_Set_Temp_22.txt'!


In [ ]:
# Cell 2: All the functions for data processing and fitting
from __future__ import annotations
from dataclasses import dataclass
from math import comb
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- All of your original helper functions go here ---
# (I've included them all below for completeness)

def _first_present(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in cols_lower:
            return cols_lower[name.lower()]
    return None

@dataclass
class ColumnMap:
    x: str
    v_real: Optional[str]
    v_imag: Optional[str]
    i_ac: Optional[str]
    freq: Optional[str]
    meas: Optional[str]

def detect_columns(df: pd.DataFrame) -> ColumnMap:
    x_candidates = [
        "H", "Field", "B", "mu0H", "magnetic_field", "field_T", "H_T",
        "Temperature (K)", "Temp", "T", "K", "temperature_K", "applied_cooling"
    ]
    x = _first_present(df, x_candidates)
    if x is None:
        numeric_cols = [c for c in df.columns if np.issubdtype(df[c].dtype, np.number)]
        if not numeric_cols:
            raise ValueError("Could not infer sweep column (field/temperature). Please specify in config.")
        x = numeric_cols[0]
    v_real = _first_present(df, ["V_real", "Vx", "in_phase", "X", "V_real_V", "V1X", "Re", "re"])
    v_imag = _first_present(df, ["V_imag", "Vy", "quadrature", "Y", "V_imag_V", "V1Y", "Im", "im"])
    i_ac = _first_present(df, ["I", "I_ac", "Iac", "I_rms", "current", "current_A", "AC_current"])
    freq = _first_present(df, ["f", "freq", "frequency", "Hz"])
    meas = _first_present(df, ["measurement", "type", "orientation", "channel", "meas", "config"])
    return ColumnMap(x=x, v_real=v_real, v_imag=v_imag, i_ac=i_ac, freq=freq, meas=meas)

def add_derived_channels(df: pd.DataFrame, cm: ColumnMap) -> pd.DataFrame:
    out = df.copy()
    if cm.v_real is not None and cm.v_imag is not None:
        out["V_real"] = pd.to_numeric(out[cm.v_real], errors="coerce")
        out["V_imag"] = pd.to_numeric(out[cm.v_imag], errors="coerce")
        out["V_mag"] = np.sqrt(out["V_real"]**2 + out["V_imag"]**2)
        out["V_phase_rad"] = np.arctan2(out["V_imag"], out["V_real"])
    elif cm.v_real is not None:
        out["V_real"] = pd.to_numeric(out[cm.v_real], errors="coerce")
        out["V_imag"] = np.nan
        out["V_mag"] = out["V_real"].abs()
        out["V_phase_rad"] = np.nan
    else:
        raise ValueError("Need at least a real voltage column. Specify in config.")
    if cm.i_ac is not None:
        out["I_ac"] = pd.to_numeric(out[cm.i_ac], errors="coerce")
    else:
        out["I_ac"] = 1.0
    out["R_real"] = out["V_real"] / out["I_ac"]
    out["R_imag"] = out["V_imag"] / out["I_ac"]
    out["R_mag"] = out["V_mag"] / out["I_ac"]
    if cm.meas is not None:
        m = out[cm.meas].astype(str).str.lower()
        m = m.replace({"rxx": "longitudinal", "rxy": "transversal", "transverse": "transversal"})
        out["_measurement_norm"] = m
    else:
        out["_measurement_norm"] = "unknown"
    out["_x"] = pd.to_numeric(out[cm.x], errors="coerce")
    return out

def bernstein_matrix(n: int, t: np.ndarray) -> np.ndarray:
    B = np.empty((t.size, n + 1), dtype=float)
    one_minus_t = 1.0 - t
    t_pows = np.vstack([t**k for k in range(n + 1)]).T
    omt_pows = np.vstack([one_minus_t**(n - k) for k in range(n + 1)]).T
    for k in range(n + 1):
        B[:, k] = comb(n, k) * t_pows[:, k] * omt_pows[:, k]
    return B

def fit_bernstein(x: np.ndarray, y: np.ndarray, degree: int = 5, l2: float = 0.0) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    idx = np.argsort(x)
    xx, yy = np.asarray(x, float)[idx], np.asarray(y, float)[idx]
    xmin, xmax = float(np.nanmin(xx)), float(np.nanmax(xx))
    if xmax <= xmin: raise ValueError("x has zero span.")
    t = (xx - xmin) / (xmax - xmin)
    mask = np.isfinite(t) & np.isfinite(yy)
    t, yy = t[mask], yy[mask]
    B = bernstein_matrix(degree, t)
    if l2 > 0:
        BTB = B.T @ B
        BTy = B.T @ yy
        ctrl_y = np.linalg.solve(BTB + l2 * np.eye(degree + 1), BTy)
    else:
        ctrl_y, *_ = np.linalg.lstsq(B, yy, rcond=None)
    t_eval = np.linspace(0, 1, 10000)
    B_eval = bernstein_matrix(degree, t_eval)
    y_fit = B_eval @ ctrl_y
    return t_eval * (xmax - xmin) + xmin, y_fit, ctrl_y

def chord_length_parameterize(points: np.ndarray) -> np.ndarray:
    diffs = np.diff(points, axis=0)
    seg = np.sqrt((diffs**2).sum(axis=1))
    s = np.concatenate([[0.0], np.cumsum(seg)])
    return s / s[-1] if s[-1] != 0 else np.linspace(0, 1, len(points))

def fit_cubic_bezier_segment(points: np.ndarray) -> np.ndarray:
    if points.shape[0] < 4: raise ValueError("Need at least 4 points for a cubic segment.")
    P0, P3 = points[0], points[-1]
    t = chord_length_parameterize(points)
    B1, B2 = 3 * t * (1 - t)**2, 3 * t**2 * (1 - t)
    M = np.column_stack([B1, B2])
    rhs = points - ((1 - t)**3)[:, None] * P0 - (t**3)[:, None] * P3
    P12x, *_ = np.linalg.lstsq(M, rhs[:, 0], rcond=None)
    P12y, *_ = np.linalg.lstsq(M, rhs[:, 1], rcond=None)
    return np.vstack([P0, [P12x[0], P12y[0]], [P12x[1], P12y[1]], P3])

def eval_cubic_bezier(CP: np.ndarray, t: np.ndarray) -> np.ndarray:
    B0, B1, B2, B3 = (1-t)**3, 3*t*(1-t)**2, 3*t**2*(1-t), t**3
    return B0[:,None]*CP[0] + B1[:,None]*CP[1] + B2[:,None]*CP[2] + B3[:,None]*CP[3]

def fit_piecewise_bezier(x: np.ndarray, y: np.ndarray, segment_len: int = 10, points_per_segment: int = 1) -> Tuple[np.ndarray, np.ndarray, List[np.ndarray]]:
    if segment_len < 4: raise ValueError("segment_len must be at least 4.")
    mask = np.isfinite(x) & np.isfinite(y)
    pts = np.column_stack([x[mask], y[mask]])
    N = pts.shape[0]
    segments, xs, ys = [], [], []
    i = 0
    while i < N - 1:
        j = min(i + segment_len, N)
        seg_pts = pts[i:j]
        if seg_pts.shape[0] < 4:
            if xs and ys: xs.append([pts[-1, 0]]); ys.append([pts[-1, 1]])
            break
        CP = fit_cubic_bezier_segment(seg_pts)
        segments.append(CP)
        t_eval = np.linspace(0, 1, points_per_segment)
        eval_pts = eval_cubic_bezier(CP, t_eval)
        if i > 0: eval_pts = eval_pts[1:]
        xs.append(eval_pts[:, 0])
        ys.append(eval_pts[:, 1])
        i = j - 1
    return np.concatenate(xs) if xs else np.array([]), np.concatenate(ys) if ys else np.array([]), segments

def make_plot(x, y, x_fit, y_fit, title, xlabel, ylabel, save_path=None):
    plt.figure(figsize=(8, 5.5))
    plt.scatter(x, y, s=15, alpha=0.7, label="Data")
    plt.plot(x_fit, y_fit, linewidth=1.0, color='orangered', label="Bézier fit")
    plt.title(title, fontsize=14)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200)
    plt.show()


# --- The main logic, now wrapped in a function ---
def run_hall_fit(config: dict):
    """
    Main function to run the Hall data fitting process based on a config dictionary.
    """
    file_path = config.get('file')
    if not file_path:
        raise ValueError("Config must include a 'file' key with the file path.")

    # Load data
    try:
        # Use a flexible separator for common text formats
        df = pd.read_csv(file_path, sep=r'\s+', engine='python')
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # Detect and process columns
    cm = detect_columns(df)
    # Override auto-detection with config values if they exist
    cm.x = config.get('xcol', cm.x)
    cm.v_real = config.get('real', cm.v_real)
    cm.v_imag = config.get('imag', cm.v_imag)
    cm.i_ac = config.get('icurr', cm.i_ac)
    cm.freq = config.get('freq', cm.freq)
    cm.meas = config.get('meas', cm.meas)

    dfp = add_derived_channels(df, cm)

    # Filter by measurement type
    measurement_type = config.get('measurement', 'all')
    if measurement_type != "all":
        dfp = dfp[dfp["_measurement_norm"] == measurement_type].copy()
        if dfp.empty:
            print(f"Warning: No data found for measurement type '{measurement_type}'.")
            return

    # Get x and y data for fitting
    y_col = config.get('y', 'R_real')
    x = dfp["_x"].to_numpy()
    y = dfp[y_col].to_numpy()

    # Perform the selected fit
    method = config.get('method', 'bernstein')
    title = f"{method.capitalize()} Bézier Fit: {y_col} vs {cm.x}"

    if method == "bernstein":
      #  degree = config.get('degree', 7)
       # l2_lambda = config.get('lambda', 0.0)
       # x_fit, y_fit, _ = fit_bernstein(x, y, degree=degree, l2=l2_lambda)
    #if method == "fit_piecewise_bezier":
        segment_len = config.get('segment_len', 1000)
        points_per_segment = config.get('points_per_segment', 1000)
        x_fit, y_fit, _ = fit_piecewise_bezier(x, y,segment_len=1000, points_per_segment=10 )

   # elif method == "fit_piecewise_bezier":
    #    x_fit, y_fit, ctrl_y = fit_piecewise_bezier(xs, ys, segment_len=segment_len, points_per_segment=points_per_segment)

    elif method == "parametric":
        seg_len = config.get('segment_len', 50)
        pts_per_seg = config.get('points_per_segment', 80)
        x_fit, y_fit, _ = fit_piecewise_bezier(x, y, segment_len=seg_len, points_per_segment=pts_per_seg)
    else:
        raise ValueError(f"Unknown method '{method}' in config.")

    # Plot the results
    save_path = config.get('plot', None)
    make_plot(x, y, x_fit, y_fit, title, xlabel=cm.x, ylabel=y_col, save_path=save_path)

    # Export fitted data
    export_path = config.get('export', None)
    if export_path:
        out_df = pd.DataFrame({cm.x: x_fit, f"{y_col}_fit": y_fit})
        out_df.to_csv(export_path, index=False)
        print(f"Saved fitted curve to: {export_path}")

In [ ]:
# Cell 3: Configuration and Execution

# Use the file name from the upload in Cell 1
# If you re-run this notebook later and don't re-upload,
# you might need to set this manually, e.g., fit_config['file'] = 'my_data.txt'
if 'file_name' not in locals() or not file_name:
    print("⚠️ Please run Cell 1 to upload a data file first!")
else:
    # --- This is your new control panel! ---
    # TODO: Adjust these settings for your specific analysis.
    fit_config = {
        'file': file_name,

        # --- Data Columns (uncomment and set if auto-detection fails) ---
        # 'xcol': 'magnetic_field',
         #'real': 'V_real_V',
         #'imag': 'V_imag_V',

        # --- Filtering ---
        'measurement': 'all',  # Options: 'all', 'longitudinal', 'transversal'
        'y': 'V_phase_rad', #'V_imag', 'V_mag', 'R_imag', 'R_mag',     # Column to fit. Options: 'V_real', 'V_imag', 'V_mag', 'R_real', 'R_imag', 'R_mag'

        # --- Fitting Method ---
        'method': 'bernstein',  # Options: 'bernstein', 'parametric'

        # --- Settings for 'bernstein' method ---
        #'degree': 7,
        #'lambda': 0.0,         # L2 regularization, 0.0 means none

        # --- Settings for 'parametric' method (for loops/hysteresis) ---
         #'segment_len': 1000,
         #'points_per_segment': 1000,

        # --- Output ---
        'plot': 'fit_plot.png', # File name to save the plot, or None
        'export': 'fit_data.csv' # File name to save the fitted data, or None
    }

    # --- Run the analysis ---
    print(f"Running fit on '{fit_config['file']}'...")
    run_hall_fit(fit_config)
    print("✅ Done!")

Running fit on 'D1_Rxy_H_1uA_Set_Temp_2.txt'...
Error loading file: [Errno 2] No such file or directory: 'D1_Rxy_H_1uA_Set_Temp_2.txt'
✅ Done!
